In [1]:
import numpy as np
from scipy.optimize import minimize

# -------------------------------------------------
# USER CONFIGURATION
# -------------------------------------------------
T        = 20.0          # investment horizon (years)
dt       = 0.25          # rebalancing interval
N        = int(T/dt)     # number of steps
K        = 50_000        # Monte-Carlo paths
I        = 3             # number of risky assets
π        = 0.10          # contribution rate (continuous)
r        = 0.03          # risk-free rate
w0       = 1.0           # initial wealth
p_min    = 0.0           # no short selling
p_max    = 1.0           # no leverage
M        = 15            # wealth grid points (coarse for speed)
W_max    = 30.0          # upper wealth bound for grid
λ_values = np.linspace(0.0, 1.5, 11)  # risk-aversion grid

In [2]:
μ = np.array([0.0795, 0.07, 0.06])          # growth rates
σ = np.array([0.15,   0.12, 0.09])          # volatilities
ρ = np.array([[1.0,  0.6, 0.2],
              [0.6,  1.0, 0.4],
              [0.2,  0.4, 1.0]])            # correlation matrix

Σ = np.diag(σ) @ ρ @ np.diag(σ)             # covariance
L = np.linalg.cholesky(Σ)                   # for path simulation

In [11]:
def simulate(W0, policy_fn, rng):
    W = np.full(K, W0, dtype=np.float64)
    Z = rng.standard_normal((N, K, I)) @ L.T
    for n in range(N):
        idx = n // skip
        t = t_grid[idx]
        p_vec = policy_fn(t, W)
        excess = μ - r
        W += W*(r*dt_sim + (p_vec @ excess)*dt_sim +
                (p_vec * Z[n]).sum(axis=1)*np.sqrt(dt_sim)) + π*dt_sim
    return W

In [12]:
from scipy.interpolate import RegularGridInterpolator

M_W = 15      # wealth grid size
M_t = 15      # time grid size  (must equal theta.shape[2])
t_grid = np.linspace(0, T, M_t)
W_grid = np.linspace(0, W_max, M_W)

def build_policy(theta):
    """
    theta :  (I, M, N) array with parameters
    returns a function (t, W) -> p(t,W)  shape (K, I)
    """
    # clip to admissible range
    theta = np.clip(theta.reshape(I, M, N), p_min, p_max)
    # enforce leverage: sum_i p_i(t,W) <= p_max
    theta = theta / (theta.sum(axis=0, keepdims=True) + 1e-12) * p_max
    theta = np.clip(theta, p_min, p_max)

    # create interpolation objects (fast on vector inputs)
    interpolators = []
    for i in range(I):
        interp = RegularGridInterpolator((t_grid, W_grid),
                                         theta[i],
                                         bounds_error=False,
                                         fill_value=theta[i, :, -1])
        interpolators.append(interp)

    def policy_fn(t, W):
        pts = np.c_[np.full_like(W, t), W]
        p = np.array([interpolators[i](pts) for i in range(I)]).T
        p = np.clip(p, p_min, p_max)
        # final leverage check
        p_sum = p.sum(axis=1, keepdims=True)
        p = np.where(p_sum > p_max, p/p_sum*p_max, p)
        return p

    return policy_fn

In [13]:
def objective(theta, λ, rng):
    pol = build_policy(theta)
    WT  = simulate(w0, pol, rng)
    EW  = WT.mean()
    diff = np.minimum(WT - EW, 0.0)
    semivar = (diff**2).mean()
    return -(EW - λ*semivar)      # scipy minimises so negate

In [14]:
rng = np.random.default_rng(42)

theta0 = np.full(I*M*N, 0.5)   # neutral start

frontier = []
for λ in λ_values:
    res = minimize(objective,
                   theta0,
                   args=(λ, rng),
                   method='L-BFGS-B',
                   bounds=[(p_min, p_max)]*len(theta0),
                   options={'maxiter': 100, 'disp': False})
    theta_star = res.x
    pol = build_policy(theta_star)
    WT = simulate(w0, pol, rng)
    EW  = WT.mean()
    sem = np.minimum(WT - EW, 0.0); sem = (sem**2).mean()
    frontier.append((EW, np.sqrt(sem)))
    theta0 = theta_star          # warm start next λ

ValueError: There are 15 points and 80 values in dimension 1